# Avance 1: exploración y preparación de datos para estimar la emisión de NOₓ

Equipo 39 · Proyecto `nox_ml` (Ciudad de México, noviembre–diciembre de 2023)

## 1. Objetivo y reglas del ejercicio

El proyecto estima **cambios de emisión de NOₓ por zona y tipo de día** combinando estaciones de monitoreo (SIMAT), el satélite TEMPO y meteorología, para separar el efecto de las emisiones del efecto del clima. Este cuaderno explora los datos que alimentan esa estimación: su estructura, calidad, relaciones y los problemas que obligan a preprocesarlos. El Avance 2 construye las características a partir de lo que aquí se decide.

Reglas que se cumplen en todo el cuaderno:

- **Las decisiones de modelado salen solo de noviembre–diciembre de 2023** (61 días). Febrero–mayo de 2024 es el periodo de prueba del reporte final: aquí aparece **únicamente en revisiones de calidad de los datos** (secciones 4.5, 5.2, 5.4 y 8.3, marcadas como *calidad*), nunca para elegir un umbral, un filtro o una característica.
- Los modelos usan las **tardes** (12–17 h locales, etiqueta SIMAT), cuando TEMPO escanea la zona y la capa límite está más desarrollada.
- El código de este cuaderno solo llama a `nox_ml` (`nox_ml.validation.report_eda` y `nox_ml.data`): no hay lógica propia, para que los números se puedan reproducir y probar.

Convención de horas: SIMAT estampa el **final** de la hora local (UTC−6, sin horario de verano) y la hora 24 llega como 00 del día siguiente. Los modelos usan el **inicio** de la hora (`utc − 1 h`), así que la etiqueta 12–17 de SIMAT equivale a las horas 11–16 en las tablas por hora.

### Fuentes

| Fuente | Qué contiene | Resolución | Unidades |
|---|---|---|---|
| SIMAT, red RAMA | NO, NO₂ y NOₓ por estación | horaria | ppb |
| SIMAT, red REDMET | viento (velocidad y dirección de dónde sopla) por estación | horaria | m/s, grados |
| ERA5 (Open-Meteo) y MERRA-2 (NASA) | altura de la capa límite | horaria, una celda | m |
| TEMPO NO₂ L2 V04 | columna troposférica de NO₂ por píxel | ~1 escaneo por hora, de 12 a 17 h | molécula/cm² |
| OpenStreetMap | calles principales (proxy de emisión vial) | vectorial | m |

Los datos viven en DVC (`dvc pull`); `eda.load_tables` lee las tablas crudas e intermedias y `load_inputs` arma las mismas entradas que ven los modelos.

In [ ]:
%matplotlib inline
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import report_eda as eda

DATA = eda.find_data()
T = eda.load_tables(DATA)
inputs = load_inputs(DATA, *eda.MODEL_PERIOD)
rows = eda.feature_rows(inputs)
hours = eda.hourly_table(inputs)
nox_pm = eda.in_window(T.nox, hours=eda.AFTERNOON)
start, end = eda.MODEL_PERIOD
print(f"Periodo de modelado: {start} a {end}; {len(hours)} horas, {len(rows):,} filas estación-hora de NOₓ")

## 2. Estructura y tipos de datos

In [ ]:
eda.structure(
    {
        "SIMAT NOₓ": T.nox,
        "SIMAT viento": T.wind,
        "Capa límite": T.pbl,
        "TEMPO (nov–dic 2023)": T.pixels,
        "TEMPO (feb–may 2024)": T.pixels_test,
    }
).round(2)

- **SIMAT NOₓ:** 376,349 filas estación-hora de 2023 y 2024 de 29 estaciones, en formato largo por estación y hora. `local` y `utc` son marcas de tiempo *sin zona horaria* a propósito (el paquete maneja UTC y el desfase de −6 h en un solo lugar, `nox_ml.time`). `id_station` es texto; NO, NO₂ y NOₓ son decimales en ppb (SIMAT las publica enteras); `flag_hi` es booleana.
- **SIMAT viento:** 342,607 filas de 26 estaciones; velocidad (`WSP`, m/s después de la corrección de la sección 5.3), dirección (`WDR`, grados desde donde sopla, 0–360), la velocidad original `WSP_raw` y dos banderas.
- **Capa límite:** 5,832 horas (nov 2023–jun 2024) indexadas por hora UTC, con la altura `pbl_m` y la fuente (`fuente`).
- **TEMPO:** cada fila es un píxel de un escaneo: columna (`vcd`), bandera de calidad (`qf`), fracción de nubes (`cf`), posición y la bandera `valid` que resume los filtros de la sección 4.5. Nov–dic 2023 tiene 165,082 píxeles de 396 escaneos; feb–may 2024, 278,998 (solo calidad).

In [ ]:
eda.column_types(T.nox)

In [ ]:
eda.column_types(T.wind)

In [ ]:
eda.column_types(T.pixels)

Cosas que se ven en los tipos y rangos y que se atienden más adelante:

- NO y NOₓ tienen 13.7 % de faltantes y NO₂ casi ninguno: un patrón estructural (sección 4.2).
- `qf` toma 4 valores distintos (0 es la única calidad que se acepta; 255 es "sin dato").
- La columna de TEMPO llega a −1.8×10¹⁷ y 9.6×10¹⁷ molécula/cm²: valores sin sentido físico que los filtros de calidad tienen que eliminar (sección 5.4).
- La velocidad del viento llega a 24.8 en `WSP_raw` pero a 13.5 en `WSP`: la corrección de mph (sección 5.3).

## 3. Estadísticos descriptivos

Todo en esta sección y en las siguientes usa las tardes (12–17 h) de noviembre–diciembre de 2023.

In [ ]:
eda.describe(nox_pm, ["NO", "NO2", "NOX"]).round(2)

El NOₓ de las tardes tiene mediana de 26 ppb y media de 30.7 ppb, con máximo de 176: una distribución **sesgada a la derecha** (sesgo 1.5; 2.7 en NO, que domina en los sitios de tráfico). Por eso el Avance 2 trabaja el objetivo en escala logarítmica. Ninguna hora llega a los 250 ppb que SIMAT marca como alta.

In [ ]:
eda.station_summary(nox_pm, T.catalog).round(1)

La diferencia entre estaciones es más grande que cualquier otra variación: la media va de 10 ppb en Montecillo (periferia) a 47 ppb en Merced (centro, tráfico), un factor de 4.6. Esa diferencia de nivel por sitio es una señal real (cercanía a calles) y no ruido; las estaciones de tráfico (MER, TLA, FAC, CAM, MGH, XAL, HGM, IZT) son las ocho más altas. Cuatro estaciones (BJU, FAR, GAM, MPA) no tienen NOₓ: se ven con 0 horas. Las estaciones con pocas horas (ACO, MON, UAX, UIZ, VIF) son las que sufrieron apagones largos (sección 4).

In [ ]:
eda.describe(hours, ["speed", "pbl", "ventilation", "vcd", "nox"]).round(2)

Resumen por hora de las variables que entran a los modelos (una fila por hora de las 366 con estaciones, viento y capa límite):

- **Viento del dominio** (promedio simple de las estaciones por hora): mediana de 1.3 m/s, sesgo a la derecha; es un viento débil, y las horas con viento < 1 m/s son las que más acumulan contaminación.
- **Capa límite:** mediana de 1,266 m y mínimo de 200 m, que es el piso que impone `load_inputs` después de promediar con la hora siguiente.
- **Ventilación** (viento × capa límite, m²/s): mediana de 1,473 y sesgo de 1.4; varía en un factor de 50 entre horas.
- **Columna TEMPO:** solo 258 de las 366 horas tienen píxeles válidos (29.5 % sin dato por nubes y filtros).

## 4. Datos faltantes y sus patrones

### 4.1 Por variable

In [ ]:
wind_pm = eda.in_window(T.wind, hours=eda.AFTERNOON)
eda.missing_by_variable(
    {"NOₓ": nox_pm, "viento": wind_pm, "capa límite": T.pbl, "TEMPO": T.pixels},
    {"NOₓ": ["NO", "NO2", "NOX"], "viento": ["WSP", "WDR"], "capa límite": ["pbl_m"], "TEMPO": ["vcd", "cf"]},
).round(2)

Los faltantes se concentran en NO y NOₓ (14.65 %), que además faltan **siempre juntos**; NO₂ nunca falta. En el viento falta el 0.1 % de la velocidad y nada de la dirección; la capa límite está completa. En TEMPO falta 2.8 % de la columna y 2.7 % de la nubosidad.

### 4.2 Patrones: qué falta y juntos con qué

In [ ]:
eda.missing_patterns(nox_pm).round(2)

Solo hay dos combinaciones: todo presente (85.4 %) o NO y NOₓ ausentes con NO₂ presente (14.6 %). Las 1,356 filas con NO y NOₓ vacíos pertenecen en su totalidad a **cuatro estaciones que solo miden NO₂**: BJU, FAR, GAM y MPA (362 + 335 + 330 + 329 filas). Es un **faltante estructural**, no aleatorio: esas estaciones nunca van a aportar NOₓ y no hay nada que imputar.

In [ ]:
eda.missing_by_station(T.nox, eda.MODEL_PERIOD).round(1)

In [ ]:
eda.plot_missing_profile(T.nox, eda.MODEL_PERIOD)

Con la medida correcta (horas esperadas, no solo filas vacías) se ve lo mismo y algo más: las cuatro estaciones sin NOₓ faltan al 100 %; tres más tienen entre 45 y 74 % de faltantes por apagones (Montecillo, Acolman y UAM Xochimilco) y otras dos cerca de 28 % (UAM Iztapalapa y Villa de las Flores). Por hora del día el faltante es de 12 a 14 % casi todo el día pero **sube a 25 % de la 1 a las 3 h**; probablemente son las calibraciones automáticas nocturnas de los analizadores (hipótesis, no se verificó con SIMAT). Cae fuera de la ventana de modelado: en las tardes (12–17 h) el faltante queda entre 12.5 y 14.5 %.

### 4.3 Patrón en el tiempo: apagones por estación

La matriz muestra, para cada estación y día, la fracción de las seis horas de la tarde con NOₓ.

In [ ]:
completeness = eda.completeness_matrix(T.nox, eda.MODEL_PERIOD)
eda.plot_completeness(completeness)

Los huecos son **bloques de días completos de una estación** (Montecillo desde mediados de noviembre, Tlalnepantla y UAM Iztapalapa en las primeras dos semanas, Villa de las Flores del 1 al 13 de diciembre, UAM Xochimilco desde el 5 de diciembre): averías y mantenimiento, no pérdida aleatoria de horas. Eso importa para decidir qué hacer con ellos: no se interpolan (inventarían un ritmo diario que no se midió) y el estimador tolera un número variable de estaciones por hora porque trabaja con cada fila de estación por separado.

In [ ]:
print("Días con menos de 10 estaciones con dato:", len(eda.network_outages(completeness)))
(completeness > 0).sum(axis=1).describe().round(1)

Cada día hay entre 19 y 25 estaciones con al menos una hora de dato (22 en promedio), y **ningún día** queda con menos de 10: no hay apagones de la red completa.

### 4.4 Capa límite: huecos y relleno con MERRA-2

In [ ]:
eda.pbl_sources(T.pbl)

In [ ]:
eda.plot_pbl(T.pbl)

La serie de ERA5 de Open-Meteo cubre noviembre–diciembre de 2023 sin huecos (1,464 horas) y **no existe en 2024**. Las 4,368 horas de enero a junio de 2024 se rellenan con MERRA-2 corregido: MERRA-2 es la única fuente horaria independiente con cobertura de ese hueco. Para modelar (nov–dic 2023) este relleno **no interviene**; sí afectará al periodo de prueba, así que se revisa su calidad aquí. Se compara con ERA5 donde ambos existen (enero–junio de 2023, los mismos meses del hueco).

In [ ]:
pbl_agreement, pbl_validation = eda.pbl_correction(DATA)
pbl_agreement.round(2)

In [ ]:
{k: round(v, 3) for k, v in pbl_validation.items()}

MERRA-2 sobreestima la capa límite de la tarde respecto de ERA5 (sesgo mediano de +125 a +448 m según la hora, correlación de 0.69 a 0.79), y la corrección multiplicativa por hora (factor de 0.81 a 0.95) quita ese sesgo medio. Fuera de muestra la corrección sí mejora el error mediano: ajustada con enero–marzo de 2023 y evaluada con abril–junio, el error es de 340 m (12 %) frente a 383 m (13 %) sin corregir, con correlación de 0.74. Lectura: el relleno sirve para el nivel medio y su error por hora es del orden de 12 %, así que los resultados de 2024 se acompañan de una sensibilidad a la capa límite. No cambia nada de lo decidido con nov–dic 2023, que usa ERA5.

Nota (actualizado el 27 de octubre de 2026, después de la entrega): la versión entregada reportaba que la corrección no mejoraba fuera de muestra (376 frente a 338 m) y señalaba un posible desfase de una hora entre MERRA-2 y ERA5. El desfase existía: el código marcaba cada valor de MERRA-2 una hora tarde. Se corrigió (`context/knowledge/desfase_merra2.md`) y las cifras de esta celda son las nuevas.

### 4.5 TEMPO: filtros de calidad (*calidad*)

Un píxel es válido si la bandera de calidad principal es 0, la fracción de nubes es menor que 0.2 y la columna existe. La tabla muestra cuántos quedan después de cada filtro y cuánto rechazaría cada uno por sí solo. Primero noviembre–diciembre de 2023 (periodo de modelado):

In [ ]:
eda.tempo_filters(T.pixels).round(1)

Queda **47.4 %** de los píxeles (78,240). Las nubes son el filtro dominante (rechazan 43.9 % por sí solas); la bandera de calidad, 19.2 %; la columna ausente, 2.8 %. Revisión de calidad con febrero–mayo de 2024 (no se usa para decidir nada):

In [ ]:
eda.tempo_filters(T.pixels_test).round(1)

En febrero–mayo de 2024 los filtros dejan 67.9 %: hay menos nubes (29.8 % frente a 43.9 %) y menos píxeles marcados (3.4 % frente a 19.2 %). Los filtros no se ajustan a ningún periodo: son los del producto de TEMPO y se aplican igual; el cambio en el porcentaje es de la atmósfera y del producto, no de una decisión.

## 5. Valores atípicos

### 5.1 NOₓ alto

Por estación, horas por encima de 250 ppb (la bandera de SIMAT), por encima de la valla de Tukey (1.5 RIC sobre el logaritmo) y con z robusta (mediana/MAD, también sobre el logaritmo) mayor que 3.5:

In [ ]:
eda.outlier_table(nox_pm)

En las tardes de nov–dic 2023 **ninguna hora llega a 250 ppb** (el QC de SIMAT marca 810 horas en 2023–2024, fuera de esta ventana). Con la valla de Tukey solo aparecen 33 valores (menos de 0.5 % de las filas), concentrados en Chalco (11) y Xalostoc (5), y solo dos pasan la z robusta (CHO y VIF). Son picos aislados dentro del rango físico (máximo de 176 ppb en Tlalnepantla), compatibles con tráfico o quemas locales. **Decisión: no se elimina ningún valor alto**; la bandera `flag_hi` se conserva solo como bandera.

### 5.2 Sensores atascados (*calidad*)

Un sensor atascado repite el mismo valor positivo más de 6 horas seguidas (los ceros no cuentan: aire limpio real). Se detectan en los datos crudos de SIMAT, antes de enmascararlos:

In [ ]:
stuck = eda.stuck_table(T.nox_raw, T.nox)
stuck[stuck["stuck_hours"] > 0].round(2)

Son 59 horas-estación en 2023–2024 (Tláhuac 29, Montecillo 14, Acolman 9, Pedregal 7): 0.06–0.3 % de las horas válidas de cada estación. Ejemplo con la racha más larga (Acolman, 9 horas con el mismo valor):

In [ ]:
eda.plot_stuck_example(T.nox_raw)

Dentro de la ventana de modelado el enmascaramiento afecta a:

In [ ]:
eda.masked_hours(T.nox_raw, T.nox, eda.MODEL_PERIOD)

In [ ]:
eda.masked_hours(T.nox_raw, T.nox, eda.TEST_PERIOD)

**Ninguna de las horas atascadas cae en nov–dic 2023 ni en feb–may 2024** (las dos series vacías). Se enmascaran de todos modos (NO, NO₂ y NOₓ pasan a NaN, porque el analizador mide NO y NOₓ y un NOₓ pegado vuelve dudosos los tres) porque es barato, evita dar peso a lecturas que no son mediciones y protege el pipeline si cambia la ventana. Pero hay que decir claro que, **en estos periodos, su efecto es nulo**.

### 5.3 Viento: velocidad de 2024 en mph

Mediana mensual de la velocidad del viento de las estaciones, como la publica SIMAT y corregida:

In [ ]:
eda.plot_wind_step(T.wind)

In [ ]:
eda.wind_unit_step(T.wind).loc["2023-09":"2024-03"].round(2)

La velocidad **se duplica de golpe el 1 de enero de 2024** (de 1.6 m/s en diciembre a 3.8 en enero) y sigue en ~2.2× todo 2024, sin ningún cambio de clima que lo explique. El factor es 2.237, el de mph a m/s: **la red empezó a publicar mph con la etiqueta de m/s**. La investigación lo contrastó con ERA5 y con el aeropuerto (METAR MMMX), que no muestran el salto. Se divide la velocidad de 2024 entre 2.237 (se conserva la original en `WSP_raw`); con eso la serie de 2024 queda al nivel de 2023 (1.7 m/s en enero). La corrección **no toca nov–dic 2023** (0 filas) pero es indispensable para el periodo de prueba: sin ella el viento, y con él la ventilación y todo el transporte, estaría sobreestimado 2.2 veces. Un detalle menor: la hora 24 del 31 de diciembre de 2024 llega con etiqueta de 2025 y la regla por año no la corrige:

In [ ]:
print("Horas de 2025 (etiqueta 00:00 del 1 de enero) sin corregir:", len(eda.wind_fix_gaps(T.wind)))

Son 19 filas (una por estación) fuera de cualquier periodo de estudio; se anota como pendiente.

### 5.4 TEMPO: columnas válidas y rechazadas (*calidad*)

In [ ]:
eda.tempo_column_summary(T.pixels)

Los píxeles rechazados contienen los extremos sin sentido (percentil 1 de −3.4×10¹⁵ y percentil 99 de 1.1×10¹⁷ molécula/cm², con mínimos de hasta −1.8×10¹⁷ y máximos de 9.6×10¹⁷), y los válidos quedan en un rango razonable (mediana de 9.1×10¹⁵; percentil 99 de 4.8×10¹⁶). Los filtros hacen su trabajo. Queda un 0.5 % de píxeles válidos con columna negativa (ruido de la estimación alrededor de una columna pequeña); se **conservan**, porque descartarlos sesgaría hacia arriba los promedios en el aire limpio.

## 6. Gráficas univariadas

In [ ]:
eda.plot_histograms(
    {
        "NOₓ (ppb)": nox_pm["NOX"],
        "viento del dominio (m/s)": hours["speed"],
        "capa límite (m)": hours["pbl"],
        "ventilación (m²/s)": hours["ventilation"],
        "columna TEMPO válida (molec/cm²)": T.pixels.loc[T.pixels["valid"], "vcd"],
    },
    log=["NOₓ (ppb)"],
)

- **NOₓ** (escala logarítmica): campana asimétrica con mediana de 26 ppb; en escala lineal tiene una cola larga a la derecha. Los valores bajos aparecen en saltos porque SIMAT publica ppb enteros.
- **Viento del dominio:** mediana de 1.3 m/s y cola hacia 4.4 m/s; casi no hay calmas absolutas.
- **Capa límite:** entre 200 (el piso) y 2,550 m, casi simétrica.
- **Ventilación:** cola larga a la derecha por la multiplicación de dos variables positivas.
- **Columna TEMPO válida** (píxel a píxel): asimétrica a la derecha, con mediana de 9.1×10¹⁵ y percentil 99 de 4.8×10¹⁶ molécula/cm².

In [ ]:
eda.plot_station_boxplots(nox_pm)

El gráfico de cajas por estación (escala logarítmica, ordenado por mediana) muestra que **el nivel del sitio** domina la dispersión: la mediana va de 9 ppb (Montecillo) a 42 ppb (Merced), con rangos intercuartílicos del mismo tamaño en casi todas las estaciones. Los puntos alejados son las colas altas de los sitios de tráfico.

## 7. Análisis bivariado y multivariado

### 7.1 NOₓ y ventilación

La física dice que, con emisión constante, la concentración es proporcional a 1 / ventilación. Se mide con las 7,537 filas estación-hora:

In [ ]:
ventilation_deciles, rho, slope = eda.ventilation_relation(rows)
print(f"Spearman = {rho:.2f}; pendiente de log NOₓ sobre log ventilación = {slope:.2f}")
eda.plot_ventilation(rows)

In [ ]:
ventilation_deciles.round(1)

La relación es clara pero **mucho más débil que 1 / ventilación**: Spearman de −0.34 y pendiente de −0.32 en log-log (con −1 sería proporcional). La mediana baja de 38 ppb en el decil de menor ventilación (472 m²/s) a 16 ppb en el de mayor (3,858 m²/s): un factor de 2.4 para un factor de 8.2 en ventilación. Razones esperables: hay un **fondo** de NOₓ que la ventilación del dominio no diluye; el viento del dominio es un promedio de estaciones y no el que ve cada sitio; y el nivel de cada estación mezcla emisiones distintas. Conclusión: la ventilación es el primer control meteorológico del NOₓ, pero **no es suficiente** (de ahí que los modelos añadan hora, día de la semana, dirección y nivel por sitio).

### 7.2 Hora del día y tipo de día

Ciclo diario (las 24 horas, todas las estaciones) por tipo de día; la franja gris es la ventana de los modelos:

In [ ]:
eda.plot_daily_cycle(eda.daily_cycle(T.nox))

In [ ]:
eda.day_type_table(rows).round(2)

- Pico de la mañana a las 8 h (92 ppb entre semana), cuando coinciden el tráfico y una capa límite aún baja; mínimo de la tarde entre las 15 y 17 h. **Los modelos usan justo el mínimo del ciclo**: la mezcla es máxima y el clima menos variable, pero la señal de emisión en ppb es la más pequeña.
- El domingo (y los festivos) es 44 % más bajo que un día hábil en las tardes (0.56) y el sábado 23 % (0.77); el sábado igualaba al día hábil en el pico de la mañana (95 frente a 92 ppb) pero baja en la tarde. Eso es lo esperado si la reducción del fin de semana es de tráfico.
- **Precaución:** estos cocientes crudos no son cocientes de emisión. El domingo hay además 9 % menos ventilación (0.91) y 6 % más el sábado (1.06): el clima de cada tipo de día es distinto, y eso es justo lo que los modelos separan de la emisión. En la muestra hay 41 días hábiles, 9 sábados y 11 domingos o festivos (20 de noviembre y 25 de diciembre cuentan como domingo).

### 7.3 TEMPO frente a las estaciones

Cada hora de estación con NO₂ se empareja con el píxel válido de TEMPO más cercano (a menos de 3 km) de la misma hora:

In [ ]:
pairs = eda.tempo_station_pairs(T)
eda.pair_correlations(pairs).round(2)

In [ ]:
eda.plot_tempo_pairs(pairs)

La columna satelital y el NO₂ de superficie se parecen: r = 0.77 en los 4,906 pares hora-estación (Spearman 0.76), 0.70 dentro de cada escaneo (mediana de 239 escaneos con al menos 5 estaciones: acuerdo **espacial**) y 0.85 entre medias de estaciones. Son las mismas cifras de la investigación. La nube de puntos se ensancha a valores altos (la columna integra toda la capa límite y la estación mide la superficie) y las medias por estación ordenan los sitios casi igual. Es la evidencia de que TEMPO aporta información espacial independiente de la red de superficie.

### 7.4 Matriz de correlación de las variables por hora

Spearman, sobre las 366 horas (promedio de estaciones por hora, columna promedio de los píxeles):

In [ ]:
columns = [
    "nox",
    "vcd",
    "speed",
    "pbl",
    "ventilation",
    "ventilation_3h",
    "ventilation_6h",
    "hour",
    "n_stations",
]
eda.plot_correlation(eda.correlation_matrix(hours, columns))

- El NOₓ de superficie se relaciona más con la **capa límite** (−0.56) que con el viento (−0.17): la dilución vertical manda en las tardes.
- La columna de TEMPO casi no depende de la capa límite (−0.10): al integrar toda la columna es poco sensible a la mezcla vertical, y por eso aporta información *distinta* de la que da la superficie.
- La ventilación acumulada de 3 y 6 horas está casi duplicada entre sí (0.99) y fuertemente ligada a la ventilación de la hora (0.70).
- El NOₓ medio de la hora se relaciona con el **número de estaciones con dato** (−0.37): cuando faltan estaciones, el promedio cambia por composición y no por clima. Otra razón para no modelar el promedio de la red, sino cada estación.

## 8. Tendencias temporales

### 8.1 Serie diaria

In [ ]:
daily = eda.daily_series(rows)
eda.plot_daily_trend(daily)

El NOₓ medio de la tarde por día va de 10 a 71 ppb, con una tendencia de dos meses: ~25 ppb a principios de noviembre y 45 ppb hacia mediados de diciembre (media móvil de 7 días). Esa alza coincide con la **baja de ventilación** de la misma época (panel inferior: de ~2,400 m²/s a principios de noviembre a ~900 a mediados de diciembre), y los sábados, domingos y festivos (cuadros rojos) quedan casi siempre por debajo de la media móvil; las excepciones son dos sábados de diciembre (2 y 23) con poca ventilación. Con solo dos meses no se puede separar tendencia estacional de variación de corto plazo: cualquier característica de "día del año" funciona aquí como un contador de días (se retoma en el Avance 2).

### 8.2 Ciclo semanal

In [ ]:
eda.plot_weekly_cycle(eda.weekly_cycle(rows))

El NOₓ es más alto el miércoles y el jueves (≈ 37.5 ppb) y cae el sábado (27) y el domingo (20). El lunes sale bajo (28) por un efecto de calendario: dos de sus ocho lunes son festivos (20 de noviembre y 25 de diciembre), que los modelos tratan como domingo. La ventilación sube y baja con el NOₓ al revés, lo que confirma que el clima y el calendario se mezclan en el ciclo semanal.

### 8.3 Calidad de los datos mes a mes (*calidad*)

Indicadores de calidad de noviembre de 2023 a mayo de 2024. Los meses de 2024 son el periodo de prueba y se muestran solo para vigilar la calidad.

In [ ]:
eda.quality_by_month(T).round(1)

La calidad es estable: de 20 a 24 estaciones reportan NOₓ cada mes y entre 16 y 20 tienen al menos 75 % de las horas de la tarde; la completitud promedio de las que reportan va de 81 a 90 %; hay estaciones de viento suficientes (10 o más) en 91–100 % de las horas; y TEMPO es válido en 42–83 % de los píxeles según la nubosidad del mes (los días de enero de 2024 no se descargaron). **Toda** la capa límite de 2024 viene de MERRA-2, mientras que la de 2023 es de ERA5: una diferencia de fuente que hay que recordar al interpretar el periodo de prueba.

## 9. Decisiones de preprocesamiento y su justificación

| # | Decisión | Por qué | Evidencia (sección) |
|---|---|---|---|
| 1 | Usar solo nov–dic 2023 para decidir; feb–may 2024 solo para revisar calidad | Es el periodo de prueba del reporte final: usarlo para elegir filtros o características contaminaría la evaluación | 1, 4.5, 5.2, 8.3 |
| 2 | Quedarse con las tardes (etiquetas SIMAT 12–17 h) | TEMPO solo escanea la zona a esas horas y la capa límite está desarrollada: es el único periodo con las tres fuentes a la vez | 1, 7.2 |
| 3 | Modelar solo estaciones con NOₓ; no imputar faltantes | Las cuatro estaciones sin NOₓ son un faltante estructural, y los huecos de las demás son bloques de días: interpolar inventaría datos | 4.1–4.3 |
| 4 | Dividir entre 2.237 la velocidad del viento de 2024 | La red publicó mph con etiqueta m/s: salto de 2.2× el 1 de enero sin causa física | 5.3 |
| 5 | Enmascarar las rachas de más de 6 horas con el mismo valor (>1 ppb) | Son lecturas de un sensor pegado, no mediciones; se anulan NO, NO₂ y NOₓ juntos | 5.2 (efecto nulo en estos periodos) |
| 6 | Capa límite: ERA5; hueco de 2024 con MERRA-2 corregido por hora; piso de 200 m | Única fuente horaria independiente para 2024; el piso evita divisiones por valores casi nulos | 4.4 (no interviene en nov–dic 2023; error de 12–14 %) |
| 7 | Píxeles TEMPO válidos: calidad 0, nubes < 0.2, columna presente; conservar columnas válidas negativas | Son los filtros del producto; las columnas negativas válidas son ruido (0.5 %) y quitarlas sesgaría el aire limpio | 4.5, 5.4 |
| 8 | No eliminar valores altos de NOₓ; `flag_hi` solo es una bandera | No hay horas sobre 250 ppb en las tardes y los picos son físicamente plausibles | 5.1 |
| 9 | Viento del dominio = promedio simple de las estaciones por hora | Punto de partida de la línea base; se prueba contra alternativas en los avances de modelos | 3, 7.1 |
| 10 | Marcas de tiempo sin zona horaria y hora de inicio (`utc − 1 h`) | Una sola convención evita desfases de una hora entre SIMAT, TEMPO y capa límite | 1, 2 |

Cuánto toca cada decisión, en números:

In [ ]:
eda.preprocessing_impact(T)

## 10. Conclusiones

1. **Los datos son suficientes y consistentes para modelar noviembre–diciembre de 2023:** 24 estaciones con NOₓ y coordenadas, 366 horas con estaciones, viento y capa límite, y 78,240 píxeles válidos de TEMPO; sin días de apagón de la red.
2. **El NOₓ depende mucho más del sitio y de la meteorología que del calendario:** el nivel por estación varía en un factor de 4.6, la capa límite explica más que el viento y la ventilación solo captura una parte de la variación (pendiente de −0.32 en lugar de −1). El domingo es 44 % más bajo que un día hábil en crudo, pero ese cociente mezcla clima y emisión.
3. **TEMPO es complementario:** correlaciona 0.77 con NO₂ de superficie (0.85 entre medias de estaciones) y es casi insensible a la capa límite, justo lo que no ven las estaciones.
4. **Los problemas de calidad conocidos están atendidos y medidos:** viento 2024 en mph (el más grave: 2.2× si no se corrige, pero fuera de nov–dic 2023), sensores atascados (59 horas, **sin efecto** en los periodos de interés), huecos de capa límite en 2024 y filtros de TEMPO.
5. **Riesgos y límites que quedan abiertos:**
   - Solo hay dos meses: no hay variación estacional que modelar ni validar, y las características anuales se comportan como un contador de días.
   - La capa límite de 2024 es MERRA-2 corregido, con un error mediano fuera de muestra de 340 m (12 %) frente a ERA5 (el desfase de una hora que tenía se corrigió después de la entrega; ver sección 4.4): el periodo de prueba necesitará una sensibilidad a la capa límite.
   - Tláhuac (TAH) mide NOₓ pero queda 400 m fuera del dominio y no entra a la inversión (24 de las 25 estaciones con NOₓ); las cuatro estaciones solo con NO₂ no aportan al estimador.
   - La corrección de mph usa el año de la etiqueta y deja sin corregir 19 filas de la hora 24 del 31 de diciembre de 2024 (fuera de los periodos de estudio).
   - Los cocientes crudos (domingo/hábil 0.56) **no** se pueden leer como cambio de emisión; para eso están los modelos del Avance 3 en adelante.

El **Avance 2** parte de estas decisiones y construye las características (ventilación y su acumulado, hora y día cíclicos, dirección del viento, tipo de día, proxy de emisión vial), justifica las transformaciones (logaritmo del objetivo, ecuación adimensional de la PINN) y evalúa qué conservar.